In [ ]:
# @title Install required dependencies
!pip install -q "crewai[tools]>=1.8.0" python-dotenv ddgs pydantic


In [ ]:
# @title Setup Gemini API Key from Colab Secrets
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    # Fallback if running outside Colab or key not in Colab Secrets
    pass


In [ ]:
# !pip install -q "crewai[tools]>=1.8.0" python-dotenv
# %pip install -U crewai crewai-tools ddgs pydantic


import os

from google.colab import files, userdata
from typing import List

from crewai import Agent, Crew, LLM, Process, Task
from crewai.flow.flow import Flow, listen, or_, router, start
from crewai.flow.human_feedback import human_feedback
from crewai.tools import BaseTool, tool
from crewai_tools import FileReadTool, FileWriterTool, ScrapeWebsiteTool
from ddgs import DDGS
from pydantic import BaseModel, Field


try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    llm = LLM(
        model="gemini/gemini-3.1-flash-lite-preview",
        api_key=userdata.get("GEMINI_API_KEY"),
        temperature=0.7,
        max_tokens=1000,
    )

    @tool("DuckDuckGo Search")
    def web_search(query: str) -> str:
        """Search the web and return titles, snippets and links."""
        results = DDGS().text(query, max_results=5)
        return "\n\n".join(
            f"{result['title']}\n{result['body']}\n{result['href']}"
            for result in results
        )

    class ReportState(BaseModel):
        topic: str = ""
        audience: str = ""
        research_notes: str = ""
        draft: str = ""
        feedback: str = ""
        final_report: str = ""

    def build_research_crew() -> Crew:
        researcher_agent = Agent(
            role="Reports Researcher",
            goal="Find reliable, recent sources about {topic}",
            backstory="You are skilled at finding credible sources quickly.",
            llm=llm,
            verbose=True,
            tools=[web_search],
            allow_delegation=False,
            max_iter=5,
        )

        analyst_agent = Agent(
            role="Research Analyst",
            goal="Extract the key facts and figures about {topic} for {audience}",
            backstory="You turn raw search results into clear findings.",
            llm=llm,
            verbose=True,
            tools=[],
            allow_delegation=False,
            max_iter=5,
        )

        checker_agent = Agent(
            role="Fact Checker",
            goal="Remove weak, duplicate or unsupported claims",
            backstory="You are strict about accuracy and always keep source links.",
            llm=llm,
            verbose=True,
            tools=[web_search],
            allow_delegation=False,
            max_iter=5,
        )

        analyst_task = Task(
            description="Search for information about {topic}. Collect sources.",
            expected_output="A list of 5 sources with short summaries and links.",
            agent=analyst_agent,
        )

        researcher_task = Task(
            description="From the sources, extract key facts for {audience}.",
            expected_output="A bullet list of key facts, each tied to a source.",
            agent=researcher_agent,
        )

        checker_task = Task(
            description="Check the facts. Drop anything weak or duplicated.",
            expected_output="A clean research notes document with sources.",
            agent=checker_agent,
        )

        return Crew(
            agents=[researcher_agent, analyst_agent, checker_agent],
            tasks=[researcher_task, analyst_task, checker_task],
            verbose=True,
        )

    def build_writing_crew() -> Crew:
        writer_agent = Agent(
            role="Report Writer",
            goal="Write a clear report for {audience}",
            backstory="You write structured, readable reports from research notes.",
            llm=llm,
            verbose=True,
            tools=[],
            allow_delegation=False,
            max_iter=5,
        )

        task = Task(
            description=(
                "Write a report on {topic} for {audience} using these notes:\n\n"
                "{research_notes}"
            ),
            expected_output="A markdown report with a title, sections and a sources list.",
            agent=writer_agent,
        )

        return Crew(
            agents=[writer_agent],
            tasks=[task],
        )

    def build_revision_crew() -> Crew:
        editor_agent = Agent(
            role="Revision Editor",
            goal="Improve the draft using only the reviewer's feedback",
            backstory="You apply feedback precisely and change nothing else.",
            llm=llm,
            verbose=True,
            tools=[],
            allow_delegation=False,
            max_iter=5,
        )

        editor_task = Task(
            description=(
                "Revise this draft.\n\nDRAFT:\n{draft}\n\n"
                "REVIEWER FEEDBACK:\n{feedback}"
            ),
            expected_output="The full revised report in markdown.",
            agent=editor_agent,
        )

        return Crew(
            agents=[editor_agent],
            tasks=[editor_task],
        )

    class ApprovalReportFlow(Flow[ReportState]):
        @start()
        def collect_topic(self):
            self.state.topic = input("Topic: ").strip()
            self.state.audience = input("Audience: ").strip()

        @listen(collect_topic)
        def run_research(self):
            result = build_research_crew().kickoff(
                inputs={
                    "topic": self.state.topic,
                    "audience": self.state.audience,
                }
            )
            self.state.research_notes = result.raw

        @listen(run_research)
        def write_report(self):
            result = build_writing_crew().kickoff(
                inputs={
                    "topic": self.state.topic,
                    "audience": self.state.audience,
                    "research_notes": self.state.research_notes,
                }
            )
            self.state.draft = result.raw

        @human_feedback(
            message="Review the draft above. Type 'approved' or write your changes:",
        )
        @listen(or_(write_report, "revised"))
        def review_draft(self):
            print(
                "\n"
                + "=" * 60
                + "\n"
                + self.state.draft
                + "\n"
                + "=" * 60
            )
            return self.state.draft

        @router(review_draft)
        def check_decision(self):
            reply = (self.last_human_feedback.feedback or "").strip()
            if reply.lower() in {"approved", "approve", "yes", "ok", "lgtm"}:
                return "approved"

            self.state.feedback = reply
            return "revise"

        @listen("approved")
        def publish_report(self):
            self.state.final_report = self.state.draft
            with open("final.md", "w", encoding="utf-8") as file:
                file.write(self.state.final_report)
            print("Saved final.md")

        @listen("revise")
        def revise_report(self):
            result = build_revision_crew().kickoff(
                inputs={
                    "draft": self.state.draft,
                    "feedback": self.state.feedback,
                }
            )
            self.state.draft = result.raw
            return "revised"

    report_flow = ApprovalReportFlow()
    result = report_flow.kickoff()
    print(result)

except Exception as error:
    print("Error:", error)
